

#  KPI Engineering & Feature Creation

### Step 1: Import Required Libraries and Load Dataset

In this step, we import the necessary Python libraries required for data analysis.

- **Pandas** → Used for data manipulation and analysis.
- **NumPy** → Used for numerical operations.

These libraries help us clean data, create new metrics, and perform calculations required for military power analysis.

In this step, we load the cleaned military dataset into Python using **Pandas**.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("military_data_cleaned.csv")

print(df.head())

       country  active_personnel  aircraft_carriers  \
0  Afghanistan           75000.0                0.0   
1      Albania            7500.0                0.0   
2      Algeria          130000.0                0.0   
3       Angola          107000.0                0.0   
4    Argentina          108000.0                0.0   

   armored_fighting_vehicles  attack_aircraft  attack_helicopters  \
0                     3902.0              0.0                 0.0   
1                     1492.0              0.0                 0.0   
2                    24920.0             42.0                74.0   
3                     2258.0              6.0                14.0   
4                    22432.0             11.0                 0.0   

   border_coverage_km  coal_consumption_mt  coal_production_cum  \
0              5987.0             503000.0             767000.0   
1               691.0             255000.0             473000.0   
2              6734.0               3000.0           

### Step 2: KPI Engineering 

#### 2.1 KPI Engineering – Assets per Capita

Here we create a new metric called **Assets per Capita**.

**Why this metric is important?**

A country may have many military assets, but if the population is extremely large, the **assets available per citizen may be low**.

**Formula Used**

Assets per Capita = Total Military Assets / Population

This KPI helps compare countries more fairly by considering population size.

In [ ]:
df["assets_per_capita"] = (
    df["total_military_aircraft"] +
    df["tanks"] +
    df["total_naval_fleet"]
) / df["total_military_manpower"].replace(0, np.nan)

#### 2.2 KPI Engineering – Defense Budget to GDP Ratio

This KPI measures how much of a country's economic output is spent on defense.

**Why this metric matters?**

Some countries spend a large portion of their economy on military power.  
This indicator helps understand **defense spending priority**.

**Formula Used**

Defense Budget to GDP Ratio = Defense Budget / GDP

This helps identify countries that prioritize military investment.

In [ ]:
df["budget_to_gdp_ratio"] = (
    df["defense_budget_usd"] /
    df["purchasing_power_parity_usd"].replace(0, np.nan)
)

### 2.3 Ranking Countries by Defense Budget

In this step, we rank countries based on their **total defense budget**.

**Why ranking is useful?**

Ranking helps quickly identify:

- Countries with the highest defense spending
- Countries with relatively smaller military budgets

This allows easy comparison of military investment across nations.

In [ ]:
df["rank"] = df["defense_budget_usd"].rank(ascending=False)

top_rank = df["rank"].min()

df["power_index_rank_gap"] = df["rank"] - top_rank

### 2.4 Power Index Rank Gap

Here we calculate the **difference between a country's rank and the top-ranked country**.

**Purpose of this KPI**

This metric helps us understand how far each country is from the **global military leader**.

A smaller gap means the country is closer to the top military power.

In [ ]:
df["rank"] = df["defense_budget_usd"].rank(ascending=False)

In [ ]:
top_rank = df["rank"].min()

In [ ]:
df["power_index_rank_gap"] = df["rank"] - top_rank

In [ ]:
df["global_rank"] = df["military_power_score"].rank(ascending=False)

### Step 3: Strength Category Classification

### Military Strength Category Classification

In this step, countries are categorized into different **strength groups** based on their military power score.

**Categories Created**

Countries are grouped into categories such as:

- Strong
- Moderate
- Developing

**Why this helps**

Instead of looking only at numbers, this classification provides an **easy visual understanding of military strength levels across nations**.

In [ ]:
df["strength_category"] = pd.cut(
    df["assets_per_capita"],
    bins=3,
    labels=["Low", "Medium", "High"]
)

### STEP 4: Validate KPI Values

Check division by zero

In [ ]:
import numpy as np

print("Infinite values check:\n")
print(np.isinf(df.select_dtypes(include=[float, int])).sum())

Infinite values check:

active_personnel             0
aircraft_carriers            0
armored_fighting_vehicles    0
attack_aircraft              0
attack_helicopters           0
                            ..
military_power_score         0
assets_per_capita            0
rank                         0
power_index_rank_gap         0
global_rank                  0
Length: 66, dtype: int64


### Step 5: KPI Statistical Summary

Here we generate summary statistics of the newly created KPIs.

This includes metrics like:

- mean
- minimum
- maximum
- standard deviation

These statistics help us understand the **overall distribution of military metrics across countries**.

In [ ]:
print("\nKPI Statistics:\n")
print(df[[
    "assets_per_capita",
    "budget_to_gdp_ratio",
    "power_index_rank_gap"
]].describe())


KPI Statistics:

       assets_per_capita  budget_to_gdp_ratio  power_index_rank_gap
count         145.000000           145.000000            145.000000
mean            0.000078             0.019467             72.000000
std             0.000117             0.029967             42.001943
min             0.000000             0.000085              0.000000
25%             0.000017             0.005833             36.000000
50%             0.000039             0.012138             72.000000
75%             0.000088             0.023780            108.000000
max             0.000705             0.308121            144.000000


In [ ]:
print("\nSample Data:\n")
print(df[[
    "country",
    "assets_per_capita",
    "budget_to_gdp_ratio",
    "power_index_rank_gap"
]].head(5))


Sample Data:

       country  assets_per_capita  budget_to_gdp_ratio  power_index_rank_gap
0  Afghanistan       3.195418e-07             0.001763                 135.0
1      Albania       3.481165e-05             0.014019                 106.0
2      Algeria       9.818001e-05             0.034582                  22.0
3       Angola       8.453833e-05             0.112134                  19.0
4    Argentina       3.119328e-05             0.000819                  96.0


### Step 6: Preview Final Dataset

Before exporting the dataset, we display a few sample rows to verify that:

- KPIs were created successfully
- Rankings were calculated correctly
- Categories were assigned properly

This step ensures data quality before saving the dataset.

In [ ]:
df.to_csv("military_final.csv", index=False)